In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install sentence-transformers xgboost pyGithub torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.4 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import tqdm
import torch.nn.functional as F
import lightgbm as lgb
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import github
from github import Github, GithubException
from urllib.request import urlopen
import re
from sklearn.model_selection import KFold

In [ ]:
ds = pd.read_csv("drive/MyDrive/monnetal/700G.csv")
ds = ds[~ds['pr_title'].str.contains('rollup', case=False, na=False)]
ds = ds[ds['uid'] <= 700]


In [ ]:
import random

# Define the range and the number of elements to pick
start_range = 1
end_range = 500
num_to_pick = 70

# Generate a list of numbers in the specified range
all_numbers = list(range(start_range, end_range + 1))

# Randomly pick unique numbers
random_numbers = random.sample(all_numbers, num_to_pick)

print(f"Randomly picked {num_to_pick} numbers from {start_range}-{end_range}:")
print(random_numbers)

Randomly picked 70 numbers from 1-500:
[274, 6, 346, 21, 433, 456, 278, 233, 76, 409, 49, 180, 415, 448, 146, 45, 464, 85, 3, 40, 68, 50, 368, 98, 205, 349, 400, 382, 22, 334, 189, 494, 385, 470, 393, 315, 200, 358, 108, 214, 258, 460, 493, 230, 64, 432, 260, 399, 59, 174, 105, 212, 47, 335, 357, 156, 18, 497, 380, 474, 226, 88, 100, 60, 57, 449, 169, 388, 239, 407]


In [ ]:
randID = [242, 451, 194, 172, 384, 260, 147, 483, 70, 331, 287, 368, 34, 405, 112, 109, 25, 261, 367, 107, 321, 246, 318, 374, 240, 179, 262, 378, 462, 443, 6, 464, 306, 58, 333, 289, 386, 99, 493, 238, 164, 190, 64, 389, 479, 33, 279, 132, 498, 226, 157, 392, 373, 189, 477, 456, 286, 489, 235, 96, 484, 494, 467, 490, 205, 236, 422, 332, 291, 171]

In [ ]:
ds2 = ds[ds['uid'].isin(randID)]
ds2.to_csv("drive/MyDrive/monnetal/testDS.csv", index=False)

NameError: name 'ds' is not defined

In [ ]:
ds3=filtered_not_in_random = ds[~ds['uid'].isin(randID)]
ds3.to_csv("drive/MyDrive/monnetal/trainDS.csv", index=False)

In [ ]:
ds = pd.read_csv("drive/MyDrive/monnetal/trainDS.csv")

In [ ]:
# @title
from sentence_transformers import SentenceTransformer

vectorizer = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# @title
gh_token = ..
auth= github.Auth.Token(gh_token)
g = github.Github(auth=auth)

def clean_html(text):
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

GITHUB_LINK_PATTERN = re.compile( rf"https://github\.com/(?P<repo>[^/]+/[^/]+)/(pull|issues)/(?P<pr>\d+)" )
COMMENT_PATTERN = re.compile(
    rf"https://github\.com/(?P<repo>[^/]+/[^/]+)/(pull|issues)/(?P<pr>\d+)#(?P<type>pullrequestreview|issuecomment)-(?P<comment_id>\d+)"
)

ANY_PATTERN = re.compile(r"https?://[^\s)>\]]+")
tag_PATTERN = re.compile(r"#(\d+)")

def scrap(url: str) -> str:
   try:
    if COMMENT_PATTERN.match(url):
        match = COMMENT_PATTERN.search(url)
        repo_name = match.group("repo")
        num = int(match.group("pr"))
        comment_id = int(match.group("comment_id"))
        repo = g.get_repo(repo_name)
        try:
            pr = repo.get_pull(num)

            return pr.get_review(comment_id).body
        except GithubException as e:
            issue = repo.get_issue(num)
            return issue.get_comment(comment_id).body

    elif GITHUB_LINK_PATTERN.match(url):
        match = GITHUB_LINK_PATTERN.search(url)
        repo_name = match.group("repo")
        num = int(match.group("pr"))
        repo = g.get_repo(repo_name)
        try:
            pr = repo.get_pull(num)
            return pr.body
        except GithubException as e:
            issue = repo.get_issue(num)
            return issue.body
    else:
        try:
            with urlopen(url) as response:
                return clean_html(response.read().decode('utf-8'))
        except Exception as e:
            # print(f"Error scraping {url}: {e}")
            return "404 no body found"
   except Exception as e:
            return "404 no body found"

In [ ]:
# @title
def feature(row, vectorizer=vectorizer):
    import re

    # ---- SAFE DEFAULTS ----
    cosine = 0.0
    rep = 0
    user_flag = 0
    code_change = 0

    # ---- SAFE TEXT FETCH ----
    def safe_scrap(x):
        try:
            if x is None:
                return ""
            return str(scrap(x) or "")
        except:
            return ""

    doc = safe_scrap(row.get('pr_link'))
    query = safe_scrap(row.get('link'))

    # ---- SAFE EMBEDDING ----
    try:
        doc_v = vectorizer.encode(doc)
        query_v = vectorizer.encode(query)
        sim = vectorizer.similarity(doc_v, query_v)
        cosine = float(sim[0][0])
    except:
        cosine = 0.0

    # ---- SAME REPO CHECK ----
    try:
        rep = 1 if str(row.get('repo', "")) in str(row.get('link', "")) else 0
    except:
        rep = 0

    # ---- EXTRACT PR INFO ----
    user1 = ""
    repo_name = None
    num = None

    match = GITHUB_LINK_PATTERN.search(str(row.get('pr_link', "")))
    if match:
        try:
            repo_name = match.group("repo")
            num = int(match.group("pr"))
        except:
            repo_name, num = None, None

    # ---- FETCH PR / ISSUE USER ----
    if repo_name and num:
        try:
            repo_obj = g.get_repo(repo_name)
            try:
                pr = repo_obj.get_pull(num)
                user1 = getattr(pr.user, "login", "")
            except:
                issue = repo_obj.get_issue(num)
                user1 = getattr(issue.user, "login", "")
        except:
            user1 = ""

    # ---- PROCESS LINK TARGET ----
    assignees = []

    link = str(row.get('link', ""))
    match2 = GITHUB_LINK_PATTERN.search(link)

    if match2:
        try:
            repo_name2 = match2.group("repo")
            num2 = int(match2.group("pr"))
            repo2 = g.get_repo(repo_name2)

            try:
                pr2 = repo2.get_pull(num2)
                assignees = [u.login for u in pr2.assignees if hasattr(u, "login")]
            except:
                issue2 = repo2.get_issue(num2)
                assignees = [u.login for u in issue2.assignees if hasattr(u, "login")]
        except:
            assignees = []

    # ---- USER MATCH FEATURE ----
    try:
        user_flag = 1 if user1 and user1 in assignees else 0
    except:
        user_flag = 0

    # ---- CODE CHANGE FEATURE ----
    commit_re = re.compile(r'https://github\.com/([^/]+)/([^/]+)/commit/([a-f0-9]{7,40})')
    m = commit_re.match(link)

    if m:
        try:
            owner, repo3, sha = m.groups()
            repo_obj = g.get_repo(f"{owner}/{repo3}")
            _ = repo_obj.get_commit(sha)

            if repo_name and repo3 == repo_name.split("/")[-1]:
                code_change = 1
        except:
            code_change = 0

    return [float(cosine), int(rep), int(user_flag), int(code_change)]

In [ ]:
# @title
def get_ngrams(text, n):
    tokens = text.lower().split()
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def ngram_overlap(q, d, n=2):
    q_ngrams = set(get_ngrams(q, n))
    d_ngrams = set(get_ngrams(d, n))

    return float(len(q_ngrams & d_ngrams)) /float( max(len(q_ngrams), 1))

def feature(row):
    import re

    # ---- SAFE DEFAULTS ----
    cosine = 0.0
    unigram = 0.0
    bigram = 0.0
    trigram = 0.0
    rep = 0
    user_flag = 0
    code_change = 0
    qlen = 0
    norm = 0
    # ---- SAFE TEXT FETCH ----
    def safe_scrap(x):
        try:
            if x is None:
                return ""
            return str(scrap(x) or "")
        except:
            return ""

    doc = safe_scrap(row.get('pr_link'))
    query = safe_scrap(row.get('link'))

    # ---- SAFE EMBEDDING ----

    try:
        doc_v = vectorizer.encode(doc)
        query_v = vectorizer.encode(query)

        norm = np.linalg.norm(query_v)
        sim = vectorizer.similarity(doc_v, query_v)
        cosine = float(sim[0][0])
    except:
        cosine = 0.0
    # ---- lexical sim ----
    try:
      unigram = ngram_overlap(query, doc, 1 )
      bigram = ngram_overlap(query, doc, 2 )
      trigram = ngram_overlap(query, doc, 3 )

    except:
      unigram = 0.0
      bigram = 0.0
      trigram = 0.0

    # ---- SAME REPO CHECK ----
    try:
        rep = 1 if str(row.get('repo', "")) in str(row.get('link', "")) else 0
    except:
        rep = 0

    # ---- EXTRACT PR INFO ----
    user1 = ""
    repo_name = None
    num = None

    match = GITHUB_LINK_PATTERN.search(str(row.get('pr_link', "")))
    if match:
        try:
            repo_name = match.group("repo")
            num = int(match.group("pr"))
        except:
            repo_name, num = None, None

    # ---- FETCH PR / ISSUE USER ----
    if repo_name and num:
        try:
            repo_obj = g.get_repo(repo_name)
            try:
                pr = repo_obj.get_pull(num)
                user1 = getattr(pr.user, "login", "")
            except:
                issue = repo_obj.get_issue(num)
                user1 = getattr(issue.user, "login", "")
        except:
            user1 = ""

    # ---- PROCESS LINK TARGET ----
    assignees = []

    link = str(row.get('link', ""))
    match2 = GITHUB_LINK_PATTERN.search(link)

    if match2:
        try:
            repo_name2 = match2.group("repo")
            num2 = int(match2.group("pr"))
            repo2 = g.get_repo(repo_name2)

            try:
                pr2 = repo2.get_pull(num2)
                assignees = [u.login for u in pr2.assignees if hasattr(u, "login")]
            except:
                issue2 = repo2.get_issue(num2)
                assignees = [u.login for u in issue2.assignees if hasattr(u, "login")]
        except:
            assignees = []

    # ---- USER MATCH FEATURE ----
    try:
        user_flag = 1 if user1 and user1 in assignees else 0
    except:
        user_flag = 0

    # ---- CODE CHANGE FEATURE ----
    commit_re = re.compile(r'https://github\.com/([^/]+)/([^/]+)/commit/([a-f0-9]{7,40})')
    m = commit_re.match(link)

    if m:
        try:
            owner, repo3, sha = m.groups()
            repo_obj = g.get_repo(f"{owner}/{repo3}")
            _ = repo_obj.get_commit(sha)

            if repo_name and repo3 == repo_name.split("/")[-1]:
                code_change = 1
        except:
            code_change = 0
    qlen = len(query.split())

    return [cosine, (rep),(user_flag), (code_change), (qlen), unigram,bigram,trigram, unigram*cosine,bigram*cosine,trigram*cosine,bigram*trigram,unigram* bigram,unigram*trigram,unigram*cosine,bigram*cosine,trigram*cosine,cosine*qlen,norm ]
    # return [(cosine), (rep),(user_flag), (code_change), (qlen), norm, unigram,bigram,trigram, unigram*cosine,bigram*cosine,trigram*cosine,unigram*bigram,unigram*trigram,norm*cosine  ]

In [ ]:
# @title
import torch
import os
from tqdm import tqdm


BACKUP_FILE = "drive/MyDrive/monnetal/extract.pt"
features = []

if os.path.exists(BACKUP_FILE):
    try:
        features = torch.load(BACKUP_FILE)
        print(f"Resuming from checkpoint: {len(features)} records loaded.")
    except Exception as e:
        print(f"Failed to load backup: {e}. Starting from scratch.")

start_idx = len(features)


try:
    for row in tqdm(ds.iloc[start_idx:].itertuples(index=False),
                    total=len(ds),
                    initial=start_idx,
                    desc="Processing Features"):

        features.append(feature(row._asdict()))

        if len(features) % 500 == 0:
            torch.save(features, BACKUP_FILE)

except KeyboardInterrupt:
    print("\nInterrupted by user. Saving progress...")
    torch.save(features, BACKUP_FILE)
except Exception as e:
    print(f"\nError occurred: {e}")
    torch.save(features, BACKUP_FILE)
    print("Progress saved to disk.")

torch.save(features, BACKUP_FILE)
print("Processingsing complete and saved.")

Failed to load backup: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn mo

Processing Features: 100%|██████████| 3068/3068 [1:07:52<00:00,  1.33s/it]

Processingsing complete and saved.


In [ ]:
# @title
import torch
import os
from tqdm import tqdm


BACKUP_FILE = "drive/MyDrive/monnetal/testext.pt"
features = []

if os.path.exists(BACKUP_FILE):
    try:
        features = torch.load(BACKUP_FILE)
        print(f"Resuming from checkpoint: {len(features)} records loaded.")
    except Exception as e:
        print(f"Failed to load backup: {e}. Starting from scratch.")

start_idx = len(features)


try:
    for row in tqdm(ds2.iloc[start_idx:].itertuples(index=False),
                    total=len(ds2),
                    initial=start_idx,
                    desc="Processing Features"):

        features.append(feature(row._asdict()))

        if len(features) % 500 == 0:
            torch.save(features, BACKUP_FILE)

except KeyboardInterrupt:
    print("\nInterrupted by user. Saving progress...")
    torch.save(features, BACKUP_FILE)
except Exception as e:
    print(f"\nError occurred: {e}")
    torch.save(features, BACKUP_FILE)
    print("Progress saved to disk.")

torch.save(features, BACKUP_FILE)
print("Processingsing complete and saved.")

Resuming from checkpoint: 356 records loaded.


Processing Features: 100%|██████████| 356/356 [00:00<?, ?it/s]

Processingsing complete and saved.


In [ ]:
def ext(feat):
  newf = []
  for i in feat:
    # newf.append([i[0],i[1],i[2],i[3]])
    newf.append([i[0],i[1],i[2],i[3],i[4],i[5],i[6],i[7]])
  return newf

In [ ]:
features = torch.load("drive/MyDrive/monnetal/extract.pt",weights_only=False)

In [ ]:
ds.groupby('uid')['rank'].transform(lambda x: x.max() - x + 1).values

array([4, 2, 1, ..., 2, 2, 1])

In [ ]:
X.shape

NameError: name 'X' is not defined

In [ ]:
feats = ext(features)
X = np.array(feats)
group = ds.groupby('uid').size().to_list()
y = ds.groupby('uid')['rank'].transform(lambda x: x.max() - x + 1).values
groups = np.repeat(np.arange(len(group)), group)


In [ ]:
param = [

# ==========================================================
# 1. BASELINE (BEST EXPECTED)
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 1000,
    "learning_rate": 0.005,

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",
    "importance_type": "gain",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 2. Smaller trees
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 1000,
    "learning_rate": 0.005,

    "num_leaves": 15,
    "max_depth": 1,

    "min_child_samples": 10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 3. Slightly slower learning
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 900, #1200
    "learning_rate": 0.0001, #0.003

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 4. More regularization
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 1000,
    "learning_rate": 0.005,

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 20,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.2,
    "reg_lambda": 5.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 5. More stochasticity
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 500,
    "learning_rate": 0.005,

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 10,

    "subsample": 0.7,
    "colsample_bytree": 0.7,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 6. Slightly deeper
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 1000,
    "learning_rate": 0.005,

    "num_leaves": 63,
    "max_depth": 6,

    "min_child_samples": 15,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 7. Conservative
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 800,
    "learning_rate": 0.003,

    "num_leaves": 15,
    "max_depth": 4,

    "min_child_samples": 20,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.1,
    "reg_lambda": 5.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 8. Slightly faster learning
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 700,
    "learning_rate": 0.01,

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 9. Stronger L1
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 1000,
    "learning_rate": 0.005,

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.5,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
},

# ==========================================================
# 10. Reduced feature sampling
# ==========================================================
{
    "random_state": 42,
    "objective": "lambdarank",
    "metric": "ndcg",

    "n_estimators": 1000,
    "learning_rate": 0.005,

    "num_leaves": 31,
    "max_depth": 5,

    "min_child_samples": 5,

    "subsample": 0.8,
    "colsample_bytree": 0.6,

    "reg_alpha": 0.05,
    "reg_lambda": 2.0,

    "feature_pre_filter": False,

    "force_col_wise": True,
    "lambdarank_norm": True,

    "boosting_type": "gbdt",

    "verbosity": -1,
    "n_jobs": -1
}

]

In [ ]:
# @title

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
from scipy.stats import kendalltau

# ==========================================================
# Metric Functions
# ==========================================================
def precision_at_1(y_true, y_pred):
    top_pred_idx = np.argmax(y_pred)
    max_rel = np.max(y_true)
    return 1.0 if y_true[top_pred_idx] == max_rel else 0.0


def kendall_tau_score(y_true, y_pred):
    true_rank = np.argsort(np.argsort(-y_true))
    pred_rank = np.argsort(np.argsort(-y_pred))
    tau, _ = kendalltau(true_rank, pred_rank)
    return tau


def mrr_score(y_true, y_pred):
    order = np.argsort(-y_pred)
    y_true_sorted = np.array(y_true)[order]

    max_rel = np.max(y_true_sorted)

    for i, rel in enumerate(y_true_sorted):
        if rel == max_rel:
            return 1.0 / (i + 1)

    return 0.0


# ==========================================================
# One Complete 10-Fold Cross Validation
# ==========================================================
def run10fold(X, y, ds, groups, param_list):

    gkf = GroupKFold(n_splits=10)

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
        print(f"\nFold {fold}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        train_group = ds.iloc[train_idx].groupby("uid").size().to_list()
        val_group = ds.iloc[val_idx].groupby("uid").size().to_list()

        model = lgb.LGBMRanker(**param_list[fold - 1])

        model.fit(
            X_train,
            y_train,
            group=train_group
        )

        y_pred = model.predict(
            X_val
        )

        # Evaluate query by query
        start_idx = 0
        fold_ndcgs = []
        fold_p1s = []
        fold_mrrs = []
        fold_taus = []

        for g in val_group:
            end_idx = start_idx + g

            y_true_group = y_val[start_idx:end_idx]
            y_pred_group = y_pred[start_idx:end_idx]

            if g < 2 or len(set(y_true_group)) == 1:
                start_idx = end_idx
                continue

            # NDCG@10
            fold_ndcgs.append(
                ndcg_score( [y_true_group],[y_pred_group], k=min(10, g))
            )

            # Precision@1
            fold_p1s.append(
                precision_at_1(y_true_group, y_pred_group)
            )

            # MRR
            fold_mrrs.append(
                mrr_score(y_true_group, y_pred_group)
            )

            # Kendall's Tau
            fold_taus.append(
                kendall_tau_score(y_true_group, y_pred_group)
            )

            start_idx = end_idx

        # Fold summary
        fold_result = {
            "fold": fold,
            "ndcg": np.mean(fold_ndcgs),
            "precision_1": np.mean(fold_p1s),
            "mrr": np.mean(fold_mrrs),
            "kendall_tau": np.mean(fold_taus) if fold_taus else np.nan
        }

        fold_results.append(fold_result)

        print(f"NDCG@10:      {fold_result['ndcg']:.4f}")
        print(f"Precision@1:  {fold_result['precision_1']:.4f}")
        print(f"MRR:          {fold_result['mrr']:.4f}")
        print(f"Kendall Tau:  {fold_result['kendall_tau']:.4f}")

    return pd.DataFrame(fold_results)


In [ ]:
# ==========================================================
# VALID Cross-Validation Evaluation
# Using ONLY:
# - NDCG@10
# - MRR
#
# Because those are the actual ranking metrics.
# The others are mostly decorative emotional support metrics.
# ==========================================================

import numpy as np
import pandas as pd

np.random.seed(42)

all_results = []

best_global_score = -np.inf
best_global_model = None

# ==========================================================
# Loop through every parameter set
# ==========================================================
for param_idx, single_param in enumerate(param):

    print("\n" + "="*70)
    print(f"🚀 TESTING PARAM SET #{param_idx+1}")
    print("="*70)

    # same parameter for all folds
    repeated_param = [single_param] * 10

    fold_df = run10fold(
        X=X,
        y=y,
        ds=ds,
        groups=groups,
        param_list=repeated_param
    )

    # ------------------------------------------------------
    # Add metadata
    # ------------------------------------------------------
    fold_df["param_id"] = param_idx + 1

    # ======================================================
    # Evaluation score using ONLY NDCG + MRR
    # ======================================================
    # fold_df["ranking_score"] = (
    #     fold_df["ndcg"] * 0.6 +
    #     fold_df["mrr"] * 0.25 +
    #     fold_df["precision_1"] * 0.1 +
    #     fold_df["kendall_tau"] * 0.05
    # )
    fold_df["ranking_score"] = (
        fold_df["ndcg"] * 0.50 +
        fold_df["mrr"] * 0.50 )
    all_results.append(fold_df)

    # ======================================================
    # Cross-validated statistics
    # ======================================================
    mean_ndcg = fold_df["ndcg"].mean()
    std_ndcg = fold_df["ndcg"].std()

    mean_mrr = fold_df["mrr"].mean()
    std_mrr = fold_df["mrr"].std()

    mean_rank_score = fold_df["ranking_score"].mean()
    std_rank_score = fold_df["ranking_score"].std()

    # ------------------------------------------------------
    # Print summary
    # ------------------------------------------------------
    print("\n📊 CROSS-VALIDATED PERFORMANCE")

    # print(f"NDCG@10 : {mean_ndcg:.4f} ± {std_ndcg:.4f}")
    # print(f"MRR      : {mean_mrr:.4f} ± {std_mrr:.4f}")
    print(f"RankScore: {mean_rank_score:.4f} ± {std_rank_score:.4f}")

    # ======================================================
    # Select BEST PARAMETER using MEAN ranking score
    # ======================================================
    if mean_rank_score > best_global_score:

        best_global_score = mean_rank_score

        best_global_model = {

            "param_id": param_idx + 1,

            "mean_ndcg": mean_ndcg,
            "std_ndcg": std_ndcg,

            "mean_mrr": mean_mrr,
            "std_mrr": std_mrr,

            "mean_rank_score": mean_rank_score,
            "std_rank_score": std_rank_score,

            "params": single_param
        }

# ==========================================================
# Combine all fold results
# ==========================================================
results_df = pd.concat(all_results, ignore_index=True)

# ==========================================================
# Create VALID summary table
# ==========================================================
summary_df = (
    results_df
    .groupby("param_id")
    .agg({

        "ndcg": ["mean", "std"],

        "mrr": ["mean", "std"],

        "ranking_score": ["mean", "std"]

    })
)

# flatten multi-index columns
summary_df.columns = [
    "_".join(col)
    for col in summary_df.columns
]

summary_df = summary_df.reset_index()

# ==========================================================
# Sort by ranking performance
# ==========================================================
summary_df = summary_df.sort_values(
    "ranking_score_mean",
    ascending=False
)

# ==========================================================
# Save results
# ==========================================================
results_df.to_csv(
    "all_param_fold_results.csv",
    index=False
)

summary_df.to_csv(
    "param_summary_results.csv",
    index=False
)

# ==========================================================
# FINAL BEST RESULT
# ==========================================================
print("\n" + "="*80)
print("🏆 BEST PARAMETER CONFIGURATION")
print("="*80)

print(f"""
Param Set : {best_global_model['param_id']}

NDCG@10 : {best_global_model['mean_ndcg']:.4f}
           ± {best_global_model['std_ndcg']:.4f}

MRR      : {best_global_model['mean_mrr']:.4f}
           ± {best_global_model['std_mrr']:.4f}

RankScore: {best_global_model['mean_rank_score']:.4f}
           ± {best_global_model['std_rank_score']:.4f}
""")

print("\n📦 Best Parameters:")
print(best_global_model["params"])

print("\n💾 Saved:")
print(" - all_param_fold_results.csv")
print(" - param_summary_results.csv")


🚀 TESTING PARAM SET #1

Fold 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


NDCG@10:      0.9142
Precision@1:  0.6441
MRR:          0.7595
Kendall Tau:  0.6200

Fold 2


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


NDCG@10:      0.9023
Precision@1:  0.5763
MRR:          0.7115
Kendall Tau:  0.4909

Fold 3


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


NDCG@10:      0.9162
Precision@1:  0.6724
MRR:          0.7585
Kendall Tau:  0.5949

Fold 9


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


NDCG@10:      0.8894
Precision@1:  0.4655
MRR:          0.6544
Kendall Tau:  0.4603

Fold 10
NDCG@10:      0.9003
Precision@1:  0.5763
MRR:          0.7156
Kendall Tau:  0.5450

📊 CROSS-VALIDATED PERFORMANCE
RankScore: 0.8278 ± 0.0239

🏆 BEST PARAMETER CONFIGURATION

Param Set : 1

NDCG@10 : 0.9134
           ± 0.0115

MRR      : 0.7451
           ± 0.0411

RankScore: 0.8292
           ± 0.0257


📦 Best Parameters:
{'random_state': 42, 'objective': 'lambdarank', 'metric': 'ndcg', 'n_estimators': 1000, 'learning_rate': 0.005, 'num_leaves': 31, 'max_depth': 5, 'min_child_samples': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.05, 'reg_lambda': 2.0, 'feature_pre_filter': False, 'force_col_wise': True, 'lambdarank_norm': True, 'boosting_type': 'gbdt', 'importance_type': 'gain', 'verbosity': -1, 'n_jobs': -1}

💾 Saved:
 - all_param_fold_results.csv
 - param_summary_results.csv


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


In [ ]:
run10fold(X, y, ds, groups, param)

In [ ]:
param[2]

{'random_state': 42,
 'objective': 'lambdarank',
 'metric': 'ndcg',
 'n_estimators': 1200,
 'learning_rate': 0.003,
 'num_leaves': 31,
 'max_depth': 5,
 'min_child_samples': 10,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'reg_alpha': 0.05,
 'reg_lambda': 2.0,
 'feature_pre_filter': False,
 'force_col_wise': True,
 'lambdarank_norm': True,
 'boosting_type': 'gbdt',
 'verbosity': -1,
 'n_jobs': -1}

In [ ]:
param[0]

{'random_state': 42,
 'objective': 'lambdarank',
 'metric': 'ndcg',
 'n_estimators': 1000,
 'learning_rate': 0.005,
 'num_leaves': 31,
 'max_depth': 5,
 'min_child_samples': 10,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'reg_alpha': 0.05,
 'reg_lambda': 2.0,
 'feature_pre_filter': False,
 'force_col_wise': True,
 'lambdarank_norm': True,
 'boosting_type': 'gbdt',
 'importance_type': 'gain',
 'verbosity': -1,
 'n_jobs': -1}

In [ ]:

model =  lgb.LGBMRanker(
         **(param[2])
      )

model.fit(X, y, group = group)

model.booster_.save_model(f"drive/MyDrive/monnetal/LGBMAf.txt", num_iteration=model.best_iteration_)

In [ ]:
# @title
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
from scipy.stats import kendalltau
from torchmetrics.retrieval import RetrievalMRR

def precision_at_1(y_true, y_pred):
    top_pred_idx = np.argmax(y_pred)
    max_rel = np.max(y_true)
    return 1.0 if y_true[top_pred_idx] == max_rel else 0.0

def kendall_tau_score(y_true, y_pred):
    true_rank = np.argsort(np.argsort(-y_true))
    pred_rank = np.argsort(np.argsort(-y_pred))
    tau, _ = kendalltau(true_rank, pred_rank)
    return tau

# def mrr_score(y_true, y_pred):
#     preds = torch.tensor(y_pred, dtype=torch.float32)

#     max_rel = np.max(y_true)
#     target = (torch.tensor(y_true) == max_rel)

#     indexes = torch.zeros(len(preds), dtype=torch.long)

#     mrr_metric = RetrievalMRR(empty_target_action="skip")
#     return mrr_metric(preds, target, indexes=indexes).item()

def mrr_score(y_true, y_pred):
    order = np.argsort(-y_pred)
    y_true_sorted = np.array(y_true)[order]

    max_rel = np.min(y_true_sorted)

    for i, rel in enumerate(y_true_sorted):
        if rel == max_rel:
            return 1.0 / (i + 1)

    return 0.0


ds = ds.sort_values("uid").reset_index(drop=True)
X = np.array(features)
grous = ds.groupby('uid').size().to_list()
y = ds.groupby('uid')['rank'].transform(lambda x: x.max() - x + 1).values

gkf = GroupKFold(n_splits=10)

fold_scores = {
    'ndcg': [],
    'precision_1': [],
    'mrr': [],
    'kendall_tau': []
}

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"\nFold {fold+1}")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Rebuild group structure
    train_group = ds.iloc[train_idx].groupby("uid").size().to_list()
    val_group = ds.iloc[val_idx].groupby("uid").size().to_list()

    # Sanity check
    assert sum(train_group) == len(X_train)
    assert sum(val_group) == len(X_val)

    import xgboost as xgb
    ranker = xgb.XGBRanker(
    objective="rank:ndcg",        # lambdarank equivalent
    eval_metric="ndcg",

    n_estimators=1000,
    learning_rate=0.001,

    max_depth=4,                 # approx for num_leaves=15
    # LightGBM num_leaves ≈ 2^depth → 2^4 = 16

    min_child_weight=10,         # similar to min_child_samples
    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.0,
    reg_lambda=1.0,

    tree_method="hist",          # efficient like force_col_wise
    n_jobs=-1,
    random_state=42
)
    ranker.fit(
        X_train,
        y_train,
        group=train_group,
        verbose=False
    )

    y_pred = model.predict(X_val, iteration_range=(0, model.best_iteration))

    # Evaluate per query
    start = 0
    fold_ndcgs = []
    fold_p1s = []
    fold_mrrs = []
    fold_taus = []

    for i, g in enumerate(val_group):
        end = start + g


        y_true_group = y_val[start:end]
        y_pred_group = y_pred[start:end]

        # Skip groups that are too small or have no variance
        if g < 2 or len(set(y_true_group)) == 1:
            start = end
            continue

        # NDCG@10
        ndcg = ndcg_score(
            np.array([y_true_group]),
            np.array([y_pred_group]),
            k=min(10, g)
        )
        fold_ndcgs.append(ndcg)

        # Precision@1
        p1 = precision_at_1(y_true_group, y_pred_group)
        fold_p1s.append(p1)

        # MRR
        mrr = mrr_score(y_true_group, y_pred_group)
        fold_mrrs.append(mrr)

        # Kendall's Tau
        tau = kendall_tau_score(y_true_group, y_pred_group)
        if not np.isnan(tau):  # Skip NaN values
            fold_taus.append(tau)

        start = end

    # Store fold averages
    if fold_ndcgs:
        fold_scores['mrr'].append(np.mean(fold_mrrs))
        fold_scores['ndcg'].append(np.mean(fold_ndcgs))
        fold_scores['precision_1'].append(np.mean(fold_p1s))
        if fold_taus:
            fold_scores['kendall_tau'].append(np.mean(fold_taus))

    print(f"NDCG@10: {np.mean(fold_ndcgs):.4f}")
    print(f"Precision@1: {np.mean(fold_p1s):.4f}")
    print(f"MRR: {np.mean(fold_mrrs):.4f}")
    if fold_taus:
        print(f"Kendall's Tau: {np.mean(fold_taus):.4f}")

# Final results
print("\n=== FINAL CROSS-VALIDATION RESULTS ===")
for metric, scores in fold_scores.items():
    if scores:
        print(f"Mean CV {metric.upper()}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
import xgboost as xgb

ranker = xgb.XGBRanker(

    objective='rank:pairwise',

    n_estimators=1000,
    learning_rate=0.001,
    max_depth=4,
    min_child_weight=1,
    gamma=0.1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    subsample=0.8,
    colsample_bytree=0.8,
    ndcg_exp_gain=True,
    tree_method='hist'

)
ranker.fit(
    X,
    y,
    group=group,

    verbose=False
)
# ranker.fit(X, y, group = group)



XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=0.8, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, feature_weights=None, gamma=0.1,
          grow_policy=None, importance_type=None, interaction_constraints=None,
          learning_rate=0.001, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
          max_leaves=None, min_child_weight=1, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=1000,
          n_jobs=None, ndcg_exp_gain=True, ...)

XGBRanker require to fit features in X, y is the rank
then when in use, we need to extract features from input side too, let say its tf_idf cosine sim and other thing

In [ ]:
test_ds = pd.read_csv(f"drive/MyDrive/monnetal/testDS.csv")
Xtests = torch.load(f"drive/MyDrive/monnetal/testext.pt")
ranker = lgb.Booster(model_file="drive/MyDrive/monnetal/LGBMAf.txt")

In [ ]:
featxx = ext(Xtests)
featx = np.array(featxx)
test_ds.reset_index(drop=True)

,uid,pr_link,repo,pr_title,link_index,link,label_word,link_count,media_type,isGithub,rank
0,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,0,https://github.com/tensorflow/tensorflow/blob/...,git_configure.bzl,4.0,NaN,NaN,1
1,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,1,https://github.com/tensorflow/tensorflow/blob/...,tools/git/gen_git_source.py,4.0,NaN,NaN,2
2,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,2,https://github.com/tensorflow/tensorflow/commi...,https://github.com/tensorflow/tensorflow/commi...,4.0,NaN,NaN,3
3,6,https://github.com/tensorflow/tensorflow/pull/...,tensorflow/tensorflow,Removed six dependency in tools/git since it b...,3,https://github.com/tensorflow/tensorflow/blob/...,https://github.com/tensorflow/tensorflow/blob/...,4.0,NaN,NaN,4
4,25,https://github.com/pytorch/pytorch/pull/60755,pytorch/pytorch,"[cuDNN v8 API] cuDNN benchmark, convolution bw...",0,https://github.com/pytorch/pytorch/issues/58414,#58414,5.0,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...
351,494,https://github.com/flutter/flutter/pull/73154,flutter/flutter,Roll Engine from 7cb464aedbf3 to 82b4ae86d69b ...,7,https://skia.googlesource.com/buildbot/+doc/ma...,https://skia.googlesource.com/buildbot/+doc/ma...,8.0,NaN,NaN,7
352,498,https://github.com/godotengine/godot/pull/56931,godotengine/godot,Enforce mult-of-4 requirements on etcpak input.,0,https://github.com/godotengine/godot/issues/49981,#49981,4.0,NaN,NaN,2
353,498,https://github.com/godotengine/godot/pull/56931,godotengine/godot,Enforce mult-of-4 requirements on etcpak input.,1,https://github.com/godotengine/godot/pull/4737...,#47370,4.0,NaN,NaN,1
354,498,https://github.com/godotengine/godot/pull/56931,godotengine/godot,Enforce mult-of-4 requirements on etcpak input.,2,https://github.com/godotengine/godot/commit/74...,"added by reduz in February, 2019",4.0,NaN,NaN,4


In [ ]:
feat = []
current_uid = None
start_idx = 0
import tqdm
for i in tqdm.tqdm(range(len(test_ds.index)), desc="LTR: Re-ranking PR links"):
    uid = test_ds.iloc[0]['uid']
    if current_uid is not None and current_uid != uid:
        if len(feat) > 0:
            scores = ranker.predict(feat)
            test_ds.loc[start_idx:i-1, "score"] = scores
            feat = []
        start_idx = i
    feat.append(featx[i])
    current_uid = uid

# Process the final batch
if len(feat) > 0:
    # scores = ranker.predict(feat, num_iteration=ranker.best_iteration)
    scores = ranker.predict(feat)
    test_ds.loc[start_idx:, "score"] = scores

test_ds['rerank'] = (
    test_ds
    .groupby("uid")['score']
    .rank(method="first", ascending=False)
    .astype("Int64")
)

out  = f"drive/MyDrive/monnetal/lastEval/10foldx.csv"

test_ds.to_csv(out, index=False)
print(f"Saved new evaluation to {out}")

LTR: Re-ranking PR links: 100%|██████████| 356/356 [00:00<00:00, 23245.82it/s]

Saved new evaluation to drive/MyDrive/monnetal/lastEval/10foldx.csv
